# Week 7 Notebook 
- overall data quality checking
- follow up to feature engineering notebook

plan
remove lat long missings
calc iqr ranges (for important columns)
remove those outliers

# Setup


In [67]:
# imports
import pandas as pd 


In [68]:
# read-in

sold_df = pd.read_csv("../data/enriched/geo_enhanced_sold_data.csv", low_memory=False)
listings_df = pd.read_csv("../data/enriched/geo_enhanced_listings_data.csv", low_memory=False)

In [69]:
listings_df.head()

,OriginalListPrice,ListingKey,CloseDate,ClosePrice,Latitude,Longitude,UnparsedAddress,PropertyType,LivingArea,ListPrice,...,placeholder_coords_flag,non_cali_coords_flag,price_ratio,price_per_sqft,days_on_market,yr_month,listing_to_contract_days,contract_to_close_days,index_right,DistrictNa
0,929000.0,1076194146,NaN,NaN,NaN,NaN,16882 Canyon Lane,Residential,1389.0,929000.0,...,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,999999.0,1076194026,NaN,NaN,NaN,NaN,8720 S 4th Avenue,Residential,2526.0,999999.0,...,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1400000.0,1076193814,NaN,NaN,33.858559,-116.542169,505 E Molino Road,Residential,2256.0,1400000.0,...,False,True,NaN,NaN,NaN,NaN,NaN,NaN,478.0,Palm Springs Unified
3,4998888.0,1076193812,NaN,NaN,NaN,NaN,3653 Halldale Avenue,ResidentialIncome,NaN,4998888.0,...,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,549000.0,1076193525,NaN,NaN,NaN,NaN,1736 N Mcdivitt Avenue,Residential,986.0,549000.0,...,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [70]:
listings_df_edit = listings_df.dropna(subset=['Latitude', 'Longitude'])

In [71]:
listings_df_edit.info()

<class 'pandas.core.frame.DataFrame'>
Index: 941336 entries, 2 to 1053810
Data columns (total 67 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   OriginalListPrice            937525 non-null  float64
 1   ListingKey                   941336 non-null  int64  
 2   CloseDate                    277873 non-null  object 
 3   ClosePrice                   254176 non-null  float64
 4   Latitude                     941336 non-null  float64
 5   Longitude                    941336 non-null  float64
 6   UnparsedAddress              938962 non-null  object 
 7   PropertyType                 941336 non-null  object 
 8   LivingArea                   823986 non-null  float64
 9   ListPrice                    938870 non-null  float64
 10  DaysOnMarket                 941336 non-null  int64  
 11  ListOfficeName               941336 non-null  object 
 12  BuyerOfficeName              264463 non-null  object 
 13  CoL

In [72]:
def iqr_outlier_flag(df, subset, multiplier= 1.5, remove=False):
    Q1 = df[subset].quantile(0.25)
    Q3 = df[subset].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - multiplier * IQR
    upper = Q3 + multiplier * IQR
    if remove:
        df = df[(df[subset] >= lower) & (df[subset] <= upper)]
    else:
        df['outlier_flag'] = (df[subset] < lower) | (df[subset] > upper)
    return df
    

In [73]:
listings_df_edit = iqr_outlier_flag(listings_df_edit, subset='OriginalListPrice', multiplier=1.5, remove=True)
listings_df_edit.info()

<class 'pandas.core.frame.DataFrame'>
Index: 877633 entries, 2 to 1053810
Data columns (total 67 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   OriginalListPrice            877633 non-null  float64
 1   ListingKey                   877633 non-null  int64  
 2   CloseDate                    263689 non-null  object 
 3   ClosePrice                   243203 non-null  float64
 4   Latitude                     877633 non-null  float64
 5   Longitude                    877633 non-null  float64
 6   UnparsedAddress              875469 non-null  object 
 7   PropertyType                 877633 non-null  object 
 8   LivingArea                   772786 non-null  float64
 9   ListPrice                    877612 non-null  float64
 10  DaysOnMarket                 877633 non-null  int64  
 11  ListOfficeName               877633 non-null  object 
 12  BuyerOfficeName              253199 non-null  object 
 13  CoL

# Full Data Cleaning / Imputation

In [74]:
listings_df.isnull().sum()

OriginalListPrice             4354
ListingKey                       0
CloseDate                   745424
ClosePrice                  777630
Latitude                    112472
                             ...  
yr_month                    745424
listing_to_contract_days    602014
contract_to_close_days      751744
index_right                 114897
DistrictNa                  114897
Length: 67, dtype: int64

# Rules for Handling NaNs: 
- OriginalListPrice: remove rows
- close date/price: keep (in listings) 
- lat/long: remove rows (stil look into getting coords from address) 
- unparsed address: remove
- livingArea: groupwise imputation
- listprice remove (but recheck after originalistprices are gone (maybe set to originallistprice if there))
- daysonmarket: groupwise imputation(?)
- office cols: dont remove but dont impute either
- AssociationFee : maybe just drop this col check data dictionary
- AttatchedGarage : impute 0 
- ParkingTotal : groupwise impute
- lotSize: groupwise impute
- streetNumber - get from unparsed address (or maybe just remove this column? )
- Lotsize - impute 
- yearbuilt - median impute? idk
- bathrooms total - impute
- city - idk
- taxeyar - what even is this
- buildingareatotal - also what is this
- bedroomstotal - impute
- fireplace - impoute 0 
- stories / levels - impute
- lotsizearea - impute (ref other lotsize col)
- 

In [75]:
listings_df.shape

(1053811, 67)

In [76]:
listings_df = listings_df.dropna(subset=['Latitude', 'Longitude'])

In [77]:
listings_df.shape

(941336, 67)

In [78]:
listings_df = listings_df.dropna(subset=['OriginalListPrice'])
listings_df.shape

(937525, 67)

In [79]:
listings_df = listings_df.dropna(subset=['UnparsedAddress'])
listings_df.shape

(935186, 67)

In [80]:
listings_df.isnull().sum()

OriginalListPrice                0
ListingKey                       0
CloseDate                   659063
ClosePrice                  682677
Latitude                         0
                             ...  
yr_month                    659063
listing_to_contract_days    533860
contract_to_close_days      665372
index_right                   2400
DistrictNa                    2400
Length: 67, dtype: int64

In [81]:
listings_df = listings_df.drop(
    columns=["TaxYear", "BuildingAreaTotal", "LotSizeDimensions", "StreetNumberNumeric", "MainLevelBedrooms"]
)  # really high null pcts/useless

In [82]:
# group of columns to use (for grouping by ) for groupwise imputation
group = ["CountyOrParish", "PropertyType", "OriginalListPrice", "LivingArea"]

groupwise_cols = ["LivingArea", "ParkingTotal", "BedroomsTotal" ]

listings_df


,OriginalListPrice,ListingKey,CloseDate,ClosePrice,Latitude,Longitude,UnparsedAddress,PropertyType,LivingArea,ListPrice,...,placeholder_coords_flag,non_cali_coords_flag,price_ratio,price_per_sqft,days_on_market,yr_month,listing_to_contract_days,contract_to_close_days,index_right,DistrictNa
2,1400000.0,1076193814,NaN,NaN,33.858559,-116.542169,505 E Molino Road,Residential,2256.0,1400000.0,...,False,True,NaN,NaN,NaN,NaN,NaN,NaN,478.0,Palm Springs Unified
8,199990.0,1076192982,NaN,NaN,32.777540,-115.553772,1826 S 4th Street,Residential,1024.0,199990.0,...,False,True,NaN,NaN,NaN,NaN,NaN,NaN,149.0,Central Union High
9,199990.0,1076192982,NaN,NaN,32.777540,-115.553772,1826 S 4th Street,Residential,1024.0,199990.0,...,False,True,NaN,NaN,NaN,NaN,NaN,NaN,150.0,El Centro Elementary
12,835000.0,1076190486,NaN,NaN,33.166112,-117.266548,4645 Cordoba Way,Residential,1444.0,835000.0,...,False,True,NaN,NaN,NaN,NaN,NaN,NaN,582.0,Vista Unified
13,1150000.0,1076190201,NaN,NaN,33.212280,-117.218628,1517 Via Pedro,Residential,3309.0,1150000.0,...,False,True,NaN,NaN,NaN,NaN,NaN,NaN,582.0,Vista Unified
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1053806,625000.0,1060129455,NaN,NaN,32.773679,-116.933649,1725 Grove Road,Land,NaN,625000.0,...,False,True,NaN,NaN,NaN,NaN,NaN,NaN,549.0,Cajon Valley Union
1053807,625000.0,1060129455,NaN,NaN,32.773679,-116.933649,1725 Grove Road,Land,NaN,625000.0,...,False,True,NaN,NaN,NaN,NaN,NaN,NaN,560.0,Grossmont Union High
1053808,409000.0,1058408504,NaN,NaN,33.207200,-116.299900,3135 Club Circle East,Residential,1671.0,465000.0,...,False,True,NaN,NaN,NaN,NaN,NaN,NaN,548.0,Borrego Springs Unified
1053809,3900000.0,1038675566,NaN,NaN,37.497693,-120.046471,665 Acs. - Mt Bullion Cut-Off Road,Land,NaN,2200000.0,...,False,True,NaN,NaN,NaN,NaN,NaN,NaN,346.0,Mariposa County Unified


note: imputing isn't needed for dashboard making . so i dont know why im doing this. but i suppose for modeling which i will do for funsies

# Cleaning (for Analysis / Dashboarding)

- remove same nulls as before

In [83]:
listings_df.isnull().sum()

OriginalListPrice                0
ListingKey                       0
CloseDate                   659063
ClosePrice                  682677
Latitude                         0
                             ...  
yr_month                    659063
listing_to_contract_days    533860
contract_to_close_days      665372
index_right                   2400
DistrictNa                    2400
Length: 62, dtype: int64

In [84]:
listings_df.head()

,OriginalListPrice,ListingKey,CloseDate,ClosePrice,Latitude,Longitude,UnparsedAddress,PropertyType,LivingArea,ListPrice,...,placeholder_coords_flag,non_cali_coords_flag,price_ratio,price_per_sqft,days_on_market,yr_month,listing_to_contract_days,contract_to_close_days,index_right,DistrictNa
2,1400000.0,1076193814,NaN,NaN,33.858559,-116.542169,505 E Molino Road,Residential,2256.0,1400000.0,...,False,True,NaN,NaN,NaN,NaN,NaN,NaN,478.0,Palm Springs Unified
8,199990.0,1076192982,NaN,NaN,32.777540,-115.553772,1826 S 4th Street,Residential,1024.0,199990.0,...,False,True,NaN,NaN,NaN,NaN,NaN,NaN,149.0,Central Union High
9,199990.0,1076192982,NaN,NaN,32.777540,-115.553772,1826 S 4th Street,Residential,1024.0,199990.0,...,False,True,NaN,NaN,NaN,NaN,NaN,NaN,150.0,El Centro Elementary
12,835000.0,1076190486,NaN,NaN,33.166112,-117.266548,4645 Cordoba Way,Residential,1444.0,835000.0,...,False,True,NaN,NaN,NaN,NaN,NaN,NaN,582.0,Vista Unified
13,1150000.0,1076190201,NaN,NaN,33.212280,-117.218628,1517 Via Pedro,Residential,3309.0,1150000.0,...,False,True,NaN,NaN,NaN,NaN,NaN,NaN,582.0,Vista Unified
